In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---------------------------------------------------------------
# 1. Carga y purgado de datos
# ---------------------------------------------------------------
datos = pd.read_csv("dataset_consumo_electrico_30000.csv")
datos = datos.dropna(axis=0)

# ---------------------------------------------------------------
# 2. Variable a predecir y conjuntos de variables
# ---------------------------------------------------------------
y = datos["consumo_mensual_kwh"]

colsReducida = ["superficie_m2", "numero_personas", "temperatura_media_c",
                "nivel_aislamiento", "paneles_solares"]
colsAmpliada = colsReducida + ["numero_electrodomesticos", "aire_acondicionado",
                               "tipo_calefaccion"]
colsNumericas = [c for c in colsAmpliada if c != "tipo_calefaccion"]

XAmpliada = datos[colsAmpliada].copy()

# ---------------------------------------------------------------
# 3. Numerización de variables
# ---------------------------------------------------------------
XAmpliada["nivel_aislamiento"] = (XAmpliada["nivel_aislamiento"]
                                  .str.strip().str.lower()
                                  .map({"bajo": 1, "medio": 2, "alto": 3}))

mapa_si_no = {"si": 1, "sí": 1, "no": 0}
for col in ["paneles_solares", "aire_acondicionado"]:
    XAmpliada[col] = XAmpliada[col].str.strip().str.lower().map(mapa_si_no)

XAmpliada["tipo_calefaccion"] = XAmpliada["tipo_calefaccion"].str.strip()


# Versión para HistGradientBoosting: tipo_calefaccion como dtype "category"
# (el modelo la trata como categórica nativa, sin one-hot)
XHGB = XAmpliada.copy()
XHGB["tipo_calefaccion"] = XHGB["tipo_calefaccion"].astype("category")

# ---------------------------------------------------------------
# 4. Un único split (70 % entrenamiento, 30 % validación)
# ---------------------------------------------------------------
train_xampliada, val_xampliada, train_y, val_y = train_test_split(
    XAmpliada, y, test_size=0.3, random_state=42)

train_xreducida = train_xampliada[colsReducida]
val_xreducida = val_xampliada[colsReducida]

# Mismas filas para HGB
train_xhgb = XHGB.loc[train_xampliada.index]
val_xhgb = XHGB.loc[val_xampliada.index]

# ---------------------------------------------------------------
# 5. Definición de modelos (se usan igual en el split y en la CV)
# ---------------------------------------------------------------
alphas = np.logspace(-3, 3, 50)

preOLS = ColumnTransformer(
    [("cat", OneHotEncoder(drop=["Ninguna"], handle_unknown="error"),
      ["tipo_calefaccion"])],
    remainder="passthrough")

preRidge = ColumnTransformer([
    ("cat", OneHotEncoder(drop=None, handle_unknown="ignore"), ["tipo_calefaccion"]),
    ("num", StandardScaler(), colsNumericas)
])

preRF = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["tipo_calefaccion"])],
    remainder="passthrough")

def nuevo_rf():
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                 n_jobs=-1, random_state=42)

def nuevo_hgb():
    return HistGradientBoostingRegressor(
        categorical_features="from_dtype",  # detecta las columnas "category"
        learning_rate=0.05,
        max_iter=500,
        max_leaf_nodes=15,
        min_samples_leaf=50,
        l2_regularization=1.0,
        early_stopping=True,   # para solo cuando deja de mejorar en validación interna
        random_state=42)

# nombre -> (modelo, X completa para CV, columnas/datos de train y val)
modelos = {
    "Lineal Reducida": (LinearRegression(),
                        XAmpliada[colsReducida], train_xreducida, val_xreducida),
    "Lineal Ampliada": (make_pipeline(preOLS, LinearRegression()),
                        XAmpliada, train_xampliada, val_xampliada),
    "Ridge  Reducida": (make_pipeline(StandardScaler(), RidgeCV(alphas=alphas)),
                        XAmpliada[colsReducida], train_xreducida, val_xreducida),
    "Ridge  Ampliada": (make_pipeline(preRidge, RidgeCV(alphas=alphas)),
                        XAmpliada, train_xampliada, val_xampliada),
    "RF     Reducida": (nuevo_rf(),
                        XAmpliada[colsReducida], train_xreducida, val_xreducida),
    "RF     Ampliada": (make_pipeline(preRF, nuevo_rf()),
                        XAmpliada, train_xampliada, val_xampliada),
    "HGB    Reducida": (nuevo_hgb(),
                        XHGB[colsReducida], train_xreducida, val_xreducida),
    "HGB    Ampliada": (nuevo_hgb(),
                        XHGB, train_xhgb, val_xhgb),
}

# ---------------------------------------------------------------
# 6. Comparación con el split 70/30
# ---------------------------------------------------------------
print("\n=== Split 70/30 (validación) ===")
filas = []
entrenados = {}
for nombre, (modelo, _, Xtr, Xva) in modelos.items():
    modelo.fit(Xtr, train_y)
    pred = modelo.predict(Xva)
    entrenados[nombre] = modelo
    filas.append({
        "Modelo": nombre,
        "R2 train": r2_score(train_y, modelo.predict(Xtr)),
        "R2 val": r2_score(val_y, pred),
        "MAE val": mean_absolute_error(val_y, pred),
    })

tabla_split = pd.DataFrame(filas).set_index("Modelo")
print(tabla_split.round(4))

hgbAmp = entrenados["HGB    Ampliada"]
print("\nHGB Ampliada: iteraciones usadas (early stopping):", hgbAmp.n_iter_)




=== Split 70/30 (validación) ===
                 R2 train  R2 val  MAE val
Modelo                                    
Lineal Reducida    0.5428  0.5435  78.6471
Lineal Ampliada    0.6218  0.6187  71.9344
Ridge  Reducida    0.5428  0.5435  78.6472
Ridge  Ampliada    0.6218  0.6187  71.9343
RF     Reducida    0.7238  0.5097  81.2274
RF     Ampliada    0.7880  0.5950  74.0081
HGB    Reducida    0.5662  0.5434  78.5590
HGB    Ampliada    0.6483  0.6185  71.8747

HGB Ampliada: iteraciones usadas (early stopping): 218
